In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEANED_DIR = Path("../data/cleaned")

print("Feature engineering started")

Feature engineering started


In [2]:
customers = pd.read_csv(CLEANED_DIR / "customers.csv")
payments = pd.read_csv(CLEANED_DIR / "payments.csv")
subscriptions = pd.read_csv(CLEANED_DIR / "subscriptions.csv")
subscription_plans = pd.read_csv(CLEANED_DIR / "subscription_plans.csv")
viewing_activity = pd.read_csv(CLEANED_DIR / "viewing_activity.csv")
content = pd.read_csv(CLEANED_DIR / "content.csv")
support_tickets = pd.read_csv(CLEANED_DIR / "support_tickets.csv")
customer_feedback = pd.read_csv(CLEANED_DIR / "customer_feedback.csv")
churn_labels = pd.read_csv(CLEANED_DIR / "churn_labels.csv")

print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.


In [3]:
print("Customers shape:", customers.shape)

print("\nUnique customers:",
      customers["customer_id"].nunique())

print("\nCustomer columns:")
print(customers.columns.tolist())

Customers shape: (8000, 11)

Unique customers: 8000

Customer columns:
['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country', 'city', 'registration_date', 'acquisition_channel', 'customer_segment', 'preferred_language']


In [4]:
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

print("Invalid registration dates:",
      customers["registration_date"].isna().sum())

Invalid registration dates: 0


In [5]:
reference_date = pd.Timestamp("2026-06-30")

customers["tenure_days"] = (
    reference_date - customers["registration_date"]
).dt.days

print("Minimum tenure days:",
      customers["tenure_days"].min())

print("Maximum tenure days:",
      customers["tenure_days"].max())

Minimum tenure days: 30
Maximum tenure days: 2737


In [6]:
customers[[
    "customer_id",
    "registration_date",
    "tenure_days"
]].head(10)

,customer_id,registration_date,tenure_days
0,CUST000744,2019-06-03,2584
1,CUST001502,2026-01-13,168
2,CUST005383,2023-06-17,1109
3,CUST004708,2019-03-09,2670
4,CUST005877,2025-12-03,209
5,CUST003762,2025-10-12,261
6,CUST007892,2025-08-31,303
7,CUST002317,2020-04-21,2261
8,CUST005609,2023-01-03,1274
9,CUST000519,2024-02-07,874


In [7]:
watch_time = viewing_activity.groupby("customer_id")[
    "watch_duration_minutes"
].sum()

print(watch_time.head())

customer_id
CUST000001    226
CUST000002    206
CUST000003     83
CUST000004    469
CUST000005    397
Name: watch_duration_minutes, dtype: int64


In [8]:
customers["total_watch_time_minutes"] = (
    customers["customer_id"].map(watch_time)
)

customers["total_watch_time_minutes"] = (
    customers["total_watch_time_minutes"].fillna(0)
)

print(customers[[
    "customer_id",
    "total_watch_time_minutes"
]].head(10))

  customer_id  total_watch_time_minutes
0  CUST000744                     266.0
1  CUST001502                     309.0
2  CUST005383                     295.0
3  CUST004708                     426.0
4  CUST005877                     286.0
5  CUST003762                     524.0
6  CUST007892                     400.0
7  CUST002317                     271.0
8  CUST005609                     508.0
9  CUST000519                     915.0


In [9]:
viewing_sessions = viewing_activity.groupby("customer_id")[
    "viewing_id"
].count()

print(viewing_sessions.head())

customer_id
CUST000001     5
CUST000002     5
CUST000003     2
CUST000004     8
CUST000005    10
Name: viewing_id, dtype: int64


In [10]:
customers["total_viewing_sessions"] = (
    customers["customer_id"].map(viewing_sessions)
)

customers["total_viewing_sessions"] = (
    customers["total_viewing_sessions"].fillna(0)
)

print(customers[[
    "customer_id",
    "total_viewing_sessions"
]].head(10))

  customer_id  total_viewing_sessions
0  CUST000744                     5.0
1  CUST001502                     5.0
2  CUST005383                     6.0
3  CUST004708                    11.0
4  CUST005877                     5.0
5  CUST003762                    11.0
6  CUST007892                     8.0
7  CUST002317                     7.0
8  CUST005609                    13.0
9  CUST000519                    18.0


In [11]:
average_completion = viewing_activity.groupby("customer_id")[
    "completion_percentage"
].mean()

print(average_completion.head())

customer_id
CUST000001    77.20
CUST000002    65.60
CUST000003    76.50
CUST000004    63.75
CUST000005    70.00
Name: completion_percentage, dtype: float64


In [12]:
customers["average_completion_percentage"] = (
    customers["customer_id"].map(average_completion)
)

customers["average_completion_percentage"] = (
    customers["average_completion_percentage"].fillna(0)
)

print(customers[[
    "customer_id",
    "average_completion_percentage"
]].head(10))

  customer_id  average_completion_percentage
0  CUST000744                      70.200000
1  CUST001502                      72.800000
2  CUST005383                      76.500000
3  CUST004708                      64.000000
4  CUST005877                      60.800000
5  CUST003762                      74.545455
6  CUST007892                      72.375000
7  CUST002317                      65.571429
8  CUST005609                      73.230769
9  CUST000519                      62.222222


In [13]:
average_rating = customer_feedback.groupby("customer_id")[
    "rating"
].mean()

print(average_rating.head())

customer_id
CUST000001    3.0
CUST000002    1.0
CUST000003    3.0
CUST000008    3.0
CUST000009    4.0
Name: rating, dtype: float64


In [14]:
customers["average_rating"] = (
    customers["customer_id"].map(average_rating)
)

customers["average_rating"] = (
    customers["average_rating"].fillna(0)
)

print(customers[[
    "customer_id",
    "average_rating"
]].head(10))

  customer_id  average_rating
0  CUST000744        3.000000
1  CUST001502        0.000000
2  CUST005383        4.000000
3  CUST004708        0.000000
4  CUST005877        0.000000
5  CUST003762        4.000000
6  CUST007892        1.500000
7  CUST002317        3.333333
8  CUST005609        0.000000
9  CUST000519        0.000000


In [15]:
feedback_count = customer_feedback.groupby("customer_id")[
    "feedback_id"
].count()

print(feedback_count.head())

customer_id
CUST000001    2
CUST000002    1
CUST000003    1
CUST000008    1
CUST000009    1
Name: feedback_id, dtype: int64


In [16]:
customers["feedback_count"] = (
    customers["customer_id"].map(feedback_count)
)

customers["feedback_count"] = (
    customers["feedback_count"].fillna(0)
)

print(customers[[
    "customer_id",
    "feedback_count"
]].head(10))

  customer_id  feedback_count
0  CUST000744             1.0
1  CUST001502             0.0
2  CUST005383             1.0
3  CUST004708             0.0
4  CUST005877             0.0
5  CUST003762             1.0
6  CUST007892             2.0
7  CUST002317             3.0
8  CUST005609             0.0
9  CUST000519             0.0


In [17]:
support_ticket_count = support_tickets.groupby("customer_id")[
    "ticket_id"
].count()

print(support_ticket_count.head())


customer_id
CUST000001    2
CUST000002    1
CUST000003    1
CUST000007    1
CUST000010    1
Name: ticket_id, dtype: int64


In [18]:
customers["support_ticket_count"] = (
    customers["customer_id"].map(support_ticket_count)
)

customers["support_ticket_count"] = (
    customers["support_ticket_count"].fillna(0)
)

print(customers[[
    "customer_id",
    "support_ticket_count"
]].head(10))

  customer_id  support_ticket_count
0  CUST000744                   0.0
1  CUST001502                   0.0
2  CUST005383                   0.0
3  CUST004708                   0.0
4  CUST005877                   0.0
5  CUST003762                   0.0
6  CUST007892                   1.0
7  CUST002317                   0.0
8  CUST005609                   1.0
9  CUST000519                   2.0


In [19]:
average_support_satisfaction = support_tickets.groupby("customer_id")[
    "customer_satisfaction_score"
].mean()

print(average_support_satisfaction.head())

customer_id
CUST000001    2.5
CUST000002    3.0
CUST000003    NaN
CUST000007    3.0
CUST000010    5.0
Name: customer_satisfaction_score, dtype: float64


In [20]:
print("Customers with support satisfaction:",
      average_support_satisfaction.notna().sum())

print("Customers without support satisfaction:",
      average_support_satisfaction.isna().sum())

Customers with support satisfaction: 3540
Customers without support satisfaction: 868


In [21]:
customers["average_support_satisfaction"] = (
    customers["customer_id"].map(average_support_satisfaction)
)

print(customers[[
    "customer_id",
    "support_ticket_count",
    "average_support_satisfaction"
]].head(10))

  customer_id  support_ticket_count  average_support_satisfaction
0  CUST000744                   0.0                           NaN
1  CUST001502                   0.0                           NaN
2  CUST005383                   0.0                           NaN
3  CUST004708                   0.0                           NaN
4  CUST005877                   0.0                           NaN
5  CUST003762                   0.0                           NaN
6  CUST007892                   1.0                           NaN
7  CUST002317                   0.0                           NaN
8  CUST005609                   1.0                           2.0
9  CUST000519                   2.0                           4.0


In [22]:
customers["has_support_satisfaction"] = (
    customers["average_support_satisfaction"].notna().astype(int)
)

print(customers[[
    "customer_id",
    "support_ticket_count",
    "average_support_satisfaction",
    "has_support_satisfaction"
]].head(10))

  customer_id  support_ticket_count  average_support_satisfaction  \
0  CUST000744                   0.0                           NaN   
1  CUST001502                   0.0                           NaN   
2  CUST005383                   0.0                           NaN   
3  CUST004708                   0.0                           NaN   
4  CUST005877                   0.0                           NaN   
5  CUST003762                   0.0                           NaN   
6  CUST007892                   1.0                           NaN   
7  CUST002317                   0.0                           NaN   
8  CUST005609                   1.0                           2.0   
9  CUST000519                   2.0                           4.0   

   has_support_satisfaction  
0                         0  
1                         0  
2                         0  
3                         0  
4                         0  
5                         0  
6                         0  


In [23]:
successful_payments = payments[
    payments["payment_status"] == "Success"
]

successful_payment_count = successful_payments.groupby("customer_id")[
    "payment_id"
].count()

print(successful_payment_count.head())

customer_id
CUST000001    15
CUST000002     6
CUST000003    15
CUST000004    14
CUST000005    14
Name: payment_id, dtype: int64


In [24]:
customers["successful_payment_count"] = (
    customers["customer_id"].map(successful_payment_count)
)

customers["successful_payment_count"] = (
    customers["successful_payment_count"].fillna(0)
)

print(customers[[
    "customer_id",
    "successful_payment_count"
]].head(10))

  customer_id  successful_payment_count
0  CUST000744                      13.0
1  CUST001502                       5.0
2  CUST005383                      14.0
3  CUST004708                      14.0
4  CUST005877                       5.0
5  CUST003762                       7.0
6  CUST007892                      10.0
7  CUST002317                      15.0
8  CUST005609                      15.0
9  CUST000519                      15.0


In [25]:
successful_payment_amount = successful_payments.groupby("customer_id")[
    "amount"
].sum()

print(successful_payment_amount.head())

customer_id
CUST000001    333.35
CUST000002     92.94
CUST000003    344.85
CUST000004     97.86
CUST000005     97.86
Name: amount, dtype: float64


In [26]:
customers["total_successful_payment_amount"] = (
    customers["customer_id"].map(successful_payment_amount)
)

customers["total_successful_payment_amount"] = (
    customers["total_successful_payment_amount"].fillna(0)
)

print(customers[[
    "customer_id",
    "total_successful_payment_amount"
]].head(10))

  customer_id  total_successful_payment_amount
0  CUST000744                           298.87
1  CUST001502                            34.95
2  CUST005383                            90.87
3  CUST004708                           216.86
4  CUST005877                            91.96
5  CUST003762                           108.43
6  CUST007892                           192.08
7  CUST002317                           104.85
8  CUST005609                           216.86
9  CUST000519                           344.85


In [27]:
failed_payments = payments[
    payments["payment_status"] == "Failed"
]

failed_payment_count = failed_payments.groupby("customer_id")[
    "payment_id"
].count()

print(failed_payment_count.head())

customer_id
CUST000004    1
CUST000005    1
CUST000007    1
CUST000009    2
CUST000011    1
Name: payment_id, dtype: int64


In [28]:
customers["failed_payment_count"] = (
    customers["customer_id"].map(failed_payment_count)
)

customers["failed_payment_count"] = (
    customers["failed_payment_count"].fillna(0)
)

print(customers[[
    "customer_id",
    "failed_payment_count"
]].head(10))

  customer_id  failed_payment_count
0  CUST000744                   2.0
1  CUST001502                   0.0
2  CUST005383                   1.0
3  CUST004708                   1.0
4  CUST005877                   1.0
5  CUST003762                   1.0
6  CUST007892                   0.0
7  CUST002317                   0.0
8  CUST005609                   0.0
9  CUST000519                   0.0


KeyError: 'status'

In [30]:
print("Subscription columns:")
print(subscriptions.columns.tolist())

Subscription columns:
['subscription_id', 'customer_id', 'plan_id', 'subscription_start_date', 'subscription_end_date', 'subscription_status', 'auto_renew', 'cancellation_date', 'cancellation_reason', 'monthly_price']


In [31]:
print(subscriptions["subscription_status"].value_counts())

subscription_status
Active          6009
Cancelled       1991
Plan Changed     875
Name: count, dtype: int64


In [34]:
subscription_count = subscriptions.groupby("customer_id")[
    "subscription_id"
].count()

print(subscription_count.head())

customer_id
CUST000001    1
CUST000002    1
CUST000003    1
CUST000004    1
CUST000005    1
Name: subscription_id, dtype: int64


In [33]:
print(subscription_count.value_counts().sort_index())

subscription_id
1    7125
2     875
Name: count, dtype: int64


In [35]:
customers["subscription_count"] = (
    customers["customer_id"].map(subscription_count)
)

customers["subscription_count"] = (
    customers["subscription_count"].fillna(0)
)

print(customers[[
    "customer_id",
    "subscription_count"
]].head(10))

  customer_id  subscription_count
0  CUST000744                   1
1  CUST001502                   1
2  CUST005383                   1
3  CUST004708                   1
4  CUST005877                   1
5  CUST003762                   1
6  CUST007892                   1
7  CUST002317                   1
8  CUST005609                   1
9  CUST000519                   2


In [36]:
plan_changed = subscriptions[
    subscriptions["subscription_status"] == "Plan Changed"
]

plan_change_count = plan_changed.groupby("customer_id")[
    "subscription_id"
].count()

print(plan_change_count.head())

customer_id
CUST000030    1
CUST000035    1
CUST000047    1
CUST000051    1
CUST000054    1
Name: subscription_id, dtype: int64


In [37]:
customers["has_plan_change"] = (
    customers["customer_id"].map(plan_change_count)
)

customers["has_plan_change"] = (
    customers["has_plan_change"].fillna(0)
)

print(customers[[
    "customer_id",
    "subscription_count",
    "has_plan_change"
]].head(10))

  customer_id  subscription_count  has_plan_change
0  CUST000744                   1              0.0
1  CUST001502                   1              0.0
2  CUST005383                   1              0.0
3  CUST004708                   1              0.0
4  CUST005877                   1              0.0
5  CUST003762                   1              0.0
6  CUST007892                   1              0.0
7  CUST002317                   1              0.0
8  CUST005609                   1              0.0
9  CUST000519                   2              1.0


In [38]:
customers["has_plan_change"] = (
    customers["has_plan_change"].astype(int)
)

print(customers[[
    "customer_id",
    "has_plan_change"
]].head(10))

  customer_id  has_plan_change
0  CUST000744                0
1  CUST001502                0
2  CUST005383                0
3  CUST004708                0
4  CUST005877                0
5  CUST003762                0
6  CUST007892                0
7  CUST002317                0
8  CUST005609                0
9  CUST000519                1


In [39]:
subscriptions["subscription_start_date"] = pd.to_datetime(
    subscriptions["subscription_start_date"],
    errors="coerce"
)

subscriptions["subscription_end_date"] = pd.to_datetime(
    subscriptions["subscription_end_date"],
    errors="coerce"
)

print("Invalid start dates:",
      subscriptions["subscription_start_date"].isna().sum())

print("Invalid end dates:",
      subscriptions["subscription_end_date"].isna().sum())

Invalid start dates: 0
Invalid end dates: 8000


In [40]:
print("Missing subscription end dates:",
      subscriptions["subscription_end_date"].isna().sum())

print("Missing cancellation dates:",
      subscriptions["cancellation_date"].isna().sum())

Missing subscription end dates: 8000
Missing cancellation dates: 6926


In [41]:
print(
    subscriptions.groupby("subscription_status")[
        "subscription_end_date"
    ].apply(lambda x: x.isna().sum())
)

subscription_status
Active          6009
Cancelled       1991
Plan Changed       0
Name: subscription_end_date, dtype: int64


In [42]:
active_subscription = subscriptions[
    subscriptions["subscription_status"] == "Active"
]

active_customer_ids = active_subscription[
    "customer_id"
].unique()

customers["is_active_subscription"] = (
    customers["customer_id"].isin(active_customer_ids).astype(int)
)

print(customers[[
    "customer_id",
    "is_active_subscription"
]].head(10))

  customer_id  is_active_subscription
0  CUST000744                       1
1  CUST001502                       1
2  CUST005383                       1
3  CUST004708                       0
4  CUST005877                       1
5  CUST003762                       1
6  CUST007892                       1
7  CUST002317                       1
8  CUST005609                       1
9  CUST000519                       1


In [43]:
customers["average_watch_time_per_session"] = (
    customers["total_watch_time_minutes"]
    / customers["total_viewing_sessions"]
)

customers["average_watch_time_per_session"] = (
    customers["average_watch_time_per_session"].fillna(0)
)

print(customers[[
    "customer_id",
    "total_watch_time_minutes",
    "total_viewing_sessions",
    "average_watch_time_per_session"
]].head(10))

  customer_id  total_watch_time_minutes  total_viewing_sessions  \
0  CUST000744                     266.0                     5.0   
1  CUST001502                     309.0                     5.0   
2  CUST005383                     295.0                     6.0   
3  CUST004708                     426.0                    11.0   
4  CUST005877                     286.0                     5.0   
5  CUST003762                     524.0                    11.0   
6  CUST007892                     400.0                     8.0   
7  CUST002317                     271.0                     7.0   
8  CUST005609                     508.0                    13.0   
9  CUST000519                     915.0                    18.0   

   average_watch_time_per_session  
0                       53.200000  
1                       61.800000  
2                       49.166667  
3                       38.727273  
4                       57.200000  
5                       47.636364  
6         

In [44]:
unique_titles_watched = viewing_activity.groupby("customer_id")[
    "content_id"
].nunique()

print(unique_titles_watched.head())

customer_id
CUST000001     5
CUST000002     5
CUST000003     2
CUST000004     8
CUST000005    10
Name: content_id, dtype: int64


In [45]:
customers["unique_titles_watched"] = (
    customers["customer_id"].map(unique_titles_watched)
)

customers["unique_titles_watched"] = (
    customers["unique_titles_watched"].fillna(0)
)

print(customers[[
    "customer_id",
    "total_viewing_sessions",
    "unique_titles_watched"
]].head(10))



  customer_id  total_viewing_sessions  unique_titles_watched
0  CUST000744                     5.0                    5.0
1  CUST001502                     5.0                    5.0
2  CUST005383                     6.0                    6.0
3  CUST004708                    11.0                   11.0
4  CUST005877                     5.0                    5.0
5  CUST003762                    11.0                   11.0
6  CUST007892                     8.0                    8.0
7  CUST002317                     7.0                    7.0
8  CUST005609                    13.0                   13.0
9  CUST000519                    18.0                   17.0


In [46]:
active_viewing_days = viewing_activity.groupby("customer_id")[
    "viewing_date"
].nunique()

print(active_viewing_days.head())

customer_id
CUST000001    5
CUST000002    5
CUST000003    2
CUST000004    8
CUST000005    9
Name: viewing_date, dtype: int64


In [47]:
customers["active_viewing_days"] = (
    customers["customer_id"].map(active_viewing_days)
)

customers["active_viewing_days"] = (
    customers["active_viewing_days"].fillna(0)
)

print(customers[[
    "customer_id",
    "total_viewing_sessions",
    "active_viewing_days"
]].head(10))

  customer_id  total_viewing_sessions  active_viewing_days
0  CUST000744                     5.0                  5.0
1  CUST001502                     5.0                  5.0
2  CUST005383                     6.0                  6.0
3  CUST004708                    11.0                 11.0
4  CUST005877                     5.0                  5.0
5  CUST003762                    11.0                 11.0
6  CUST007892                     8.0                  8.0
7  CUST002317                     7.0                  7.0
8  CUST005609                    13.0                 13.0
9  CUST000519                    18.0                 18.0


In [48]:
print(subscriptions["plan_id"].value_counts())


plan_id
PLAN01    3671
PLAN02    3301
PLAN03    1903
Name: count, dtype: int64


In [49]:
subscriptions = subscriptions.sort_values(
    "subscription_start_date"
)

print(
    subscriptions[[
        "customer_id",
        "plan_id",
        "subscription_start_date",
        "subscription_status"
    ]].head(10)
)

     customer_id plan_id subscription_start_date subscription_status
2969  CUST002692  PLAN02              2019-01-01              Active
4301  CUST003898  PLAN01              2019-01-01              Active
6933  CUST006257  PLAN01              2019-01-01              Active
5369  CUST004857  PLAN02              2019-01-01              Active
8858  CUST007986  PLAN01              2019-01-01              Active
8359  CUST007538  PLAN03              2019-01-02              Active
5920  CUST005348  PLAN01              2019-01-03              Active
7195  CUST006491  PLAN01              2019-01-03              Active
3074  CUST002784  PLAN01              2019-01-03              Active
1064  CUST000972  PLAN03              2019-01-03              Active


In [50]:
latest_subscriptions = subscriptions.drop_duplicates(
    subset=["customer_id"],
    keep="last"
).copy()

print(latest_subscriptions[[
    "customer_id",
    "plan_id",
    "subscription_start_date",
    "subscription_status"
]].head(10))

     customer_id plan_id subscription_start_date subscription_status
2969  CUST002692  PLAN02              2019-01-01              Active
4301  CUST003898  PLAN01              2019-01-01              Active
6933  CUST006257  PLAN01              2019-01-01              Active
5369  CUST004857  PLAN02              2019-01-01              Active
8858  CUST007986  PLAN01              2019-01-01              Active
8359  CUST007538  PLAN03              2019-01-02              Active
5920  CUST005348  PLAN01              2019-01-03              Active
7195  CUST006491  PLAN01              2019-01-03              Active
3074  CUST002784  PLAN01              2019-01-03              Active
1064  CUST000972  PLAN03              2019-01-03              Active


In [51]:
multiple_subscriptions = subscription_count[
    subscription_count > 1
]

print("Customers with multiple subscriptions:",
      len(multiple_subscriptions))

print("\nSample customers:")
print(multiple_subscriptions.head(10))

Customers with multiple subscriptions: 875

Sample customers:
customer_id
CUST000030    2
CUST000035    2
CUST000047    2
CUST000051    2
CUST000054    2
CUST000055    2
CUST000064    2
CUST000072    2
CUST000085    2
CUST000087    2
Name: subscription_id, dtype: int64


In [52]:
latest_subscriptions = subscriptions.drop_duplicates(
    subset=["customer_id"],
    keep="last"
).copy()

print(latest_subscriptions[[
    "customer_id",
    "plan_id",
    "subscription_start_date",
    "subscription_status"
]].head(10))

     customer_id plan_id subscription_start_date subscription_status
2969  CUST002692  PLAN02              2019-01-01              Active
4301  CUST003898  PLAN01              2019-01-01              Active
6933  CUST006257  PLAN01              2019-01-01              Active
5369  CUST004857  PLAN02              2019-01-01              Active
8858  CUST007986  PLAN01              2019-01-01              Active
8359  CUST007538  PLAN03              2019-01-02              Active
5920  CUST005348  PLAN01              2019-01-03              Active
7195  CUST006491  PLAN01              2019-01-03              Active
3074  CUST002784  PLAN01              2019-01-03              Active
1064  CUST000972  PLAN03              2019-01-03              Active


In [53]:
current_plan = latest_subscriptions.set_index(
    "customer_id"
)["plan_id"]

customers["current_plan"] = (
    customers["customer_id"].map(current_plan)
)

print(customers[[
    "customer_id",
    "current_plan"
]].head(10))

  customer_id current_plan
0  CUST000744       PLAN03
1  CUST001502       PLAN01
2  CUST005383       PLAN01
3  CUST004708       PLAN02
4  CUST005877       PLAN03
5  CUST003762       PLAN02
6  CUST007892       PLAN02
7  CUST002317       PLAN01
8  CUST005609       PLAN02
9  CUST000519       PLAN03


In [54]:
total_payment_count = (
    customers["successful_payment_count"]
    + customers["failed_payment_count"]
)

customers["payment_failure_rate"] = (
    customers["failed_payment_count"]
    / total_payment_count
)

customers["payment_failure_rate"] = (
    customers["payment_failure_rate"].fillna(0)
)

print(customers[[
    "customer_id",
    "successful_payment_count",
    "failed_payment_count",
    "payment_failure_rate"
]].head(10))

  customer_id  successful_payment_count  failed_payment_count  \
0  CUST000744                      13.0                   2.0   
1  CUST001502                       5.0                   0.0   
2  CUST005383                      14.0                   1.0   
3  CUST004708                      14.0                   1.0   
4  CUST005877                       5.0                   1.0   
5  CUST003762                       7.0                   1.0   
6  CUST007892                      10.0                   0.0   
7  CUST002317                      15.0                   0.0   
8  CUST005609                      15.0                   0.0   
9  CUST000519                      15.0                   0.0   

   payment_failure_rate  
0              0.133333  
1              0.000000  
2              0.066667  
3              0.066667  
4              0.166667  
5              0.125000  
6              0.000000  
7              0.000000  
8              0.000000  
9              0.000000 

In [55]:
viewing_with_content = viewing_activity.merge(
    content[["content_id", "genre"]],
    on="content_id",
    how="left"
)

print(viewing_with_content[[
    "customer_id",
    "content_id",
    "genre"
]].head(10))

  customer_id content_id            genre
0  CUST000001   CNT00299           Comedy
1  CUST000001   CNT00098         Thriller
2  CUST000001   CNT00187           Action
3  CUST000001   CNT00126            Anime
4  CUST000001   CNT00179          Romance
5  CUST000002   CNT00487  Stand-Up Comedy
6  CUST000002   CNT00256            Anime
7  CUST000002   CNT00157          Fantasy
8  CUST000002   CNT00009         Thriller
9  CUST000002   CNT00211           Sports


In [56]:
genre_counts = (
    viewing_with_content.groupby(
        ["customer_id", "genre"]
    )
    .size()
    .reset_index(name="watch_count")
)

print(genre_counts.head(10))

  customer_id            genre  watch_count
0  CUST000001           Action            1
1  CUST000001            Anime            1
2  CUST000001           Comedy            1
3  CUST000001          Romance            1
4  CUST000001         Thriller            1
5  CUST000002            Anime            1
6  CUST000002          Fantasy            1
7  CUST000002           Sports            1
8  CUST000002  Stand-Up Comedy            1
9  CUST000002         Thriller            1


In [57]:
max_genre_count = genre_counts.groupby("customer_id")[
    "watch_count"
].max()

print(max_genre_count.head(10))

customer_id
CUST000001    1
CUST000002    1
CUST000003    1
CUST000004    2
CUST000005    2
CUST000006    2
CUST000007    2
CUST000008    2
CUST000009    3
CUST000010    3
Name: watch_count, dtype: int64


In [58]:
genre_counts = genre_counts.sort_values(
    ["customer_id", "watch_count", "genre"],
    ascending=[True, False, True]
)

favorite_genre = genre_counts.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)

print(favorite_genre[[
    "customer_id",
    "genre",
    "watch_count"
]].head(10))

   customer_id    genre  watch_count
0   CUST000001   Action            1
5   CUST000002    Anime            1
10  CUST000003   Action            1
15  CUST000004  Fantasy            2
19  CUST000005    Anime            2
28  CUST000006    Anime            2
35  CUST000007   Horror            2
39  CUST000008    Anime            2
47  CUST000009   Comedy            3
57  CUST000010    Drama            3


In [59]:
favorite_genre_map = favorite_genre.set_index(
    "customer_id"
)["genre"]

customers["favorite_genre"] = (
    customers["customer_id"].map(favorite_genre_map)
)

customers["favorite_genre"] = (
    customers["favorite_genre"].fillna("Unknown")
)

print(customers[[
    "customer_id",
    "favorite_genre"
]].head(10))

  customer_id favorite_genre
0  CUST000744         Action
1  CUST001502         Sports
2  CUST005383         Comedy
3  CUST004708    Documentary
4  CUST005877         Action
5  CUST003762          Drama
6  CUST007892         Comedy
7  CUST002317         Horror
8  CUST005609         Horror
9  CUST000519         Action


In [60]:
unique_genres_watched = viewing_with_content.groupby(
    "customer_id"
)["genre"].nunique()

customers["unique_genres_watched"] = (
    customers["customer_id"].map(unique_genres_watched)
)

customers["unique_genres_watched"] = (
    customers["unique_genres_watched"].fillna(0)
)

print(customers[[
    "customer_id",
    "favorite_genre",
    "unique_genres_watched"
]].head(10))

  customer_id favorite_genre  unique_genres_watched
0  CUST000744         Action                    5.0
1  CUST001502         Sports                    4.0
2  CUST005383         Comedy                    6.0
3  CUST004708    Documentary                    8.0
4  CUST005877         Action                    5.0
5  CUST003762          Drama                    8.0
6  CUST007892         Comedy                    6.0
7  CUST002317         Horror                    5.0
8  CUST005609         Horror                    9.0
9  CUST000519         Action                   11.0


In [61]:
customers["unique_genres_watched"] = (
    customers["unique_genres_watched"].astype(int)
)

print(customers[[
    "customer_id",
    "unique_genres_watched"
]].head(10))

  customer_id  unique_genres_watched
0  CUST000744                      5
1  CUST001502                      4
2  CUST005383                      6
3  CUST004708                      8
4  CUST005877                      5
5  CUST003762                      8
6  CUST007892                      6
7  CUST002317                      5
8  CUST005609                      9
9  CUST000519                     11


In [62]:
reference_date = pd.Timestamp("2026-06-30")

latest_subscriptions["subscription_start_date"] = pd.to_datetime(
    latest_subscriptions["subscription_start_date"],
    errors="coerce"
)

latest_subscriptions["current_subscription_tenure_days"] = (
    reference_date
    - latest_subscriptions["subscription_start_date"]
).dt.days

print(latest_subscriptions[[
    "customer_id",
    "plan_id",
    "subscription_start_date",
    "current_subscription_tenure_days"
]].head(10))

     customer_id plan_id subscription_start_date  \
2969  CUST002692  PLAN02              2019-01-01   
4301  CUST003898  PLAN01              2019-01-01   
6933  CUST006257  PLAN01              2019-01-01   
5369  CUST004857  PLAN02              2019-01-01   
8858  CUST007986  PLAN01              2019-01-01   
8359  CUST007538  PLAN03              2019-01-02   
5920  CUST005348  PLAN01              2019-01-03   
7195  CUST006491  PLAN01              2019-01-03   
3074  CUST002784  PLAN01              2019-01-03   
1064  CUST000972  PLAN03              2019-01-03   

      current_subscription_tenure_days  
2969                              2737  
4301                              2737  
6933                              2737  
5369                              2737  
8858                              2737  
8359                              2736  
5920                              2735  
7195                              2735  
3074                              2735  
1064             

In [63]:
subscription_tenure_map = latest_subscriptions.set_index(
    "customer_id"
)["current_subscription_tenure_days"]

customers["current_subscription_tenure_days"] = (
    customers["customer_id"].map(subscription_tenure_map)
)

customers["current_subscription_tenure_days"] = (
    customers["current_subscription_tenure_days"].fillna(0)
)

print(customers[[
    "customer_id",
    "current_plan",
    "current_subscription_tenure_days"
]].head(10))

  customer_id current_plan  current_subscription_tenure_days
0  CUST000744       PLAN03                              2584
1  CUST001502       PLAN01                               168
2  CUST005383       PLAN01                              1109
3  CUST004708       PLAN02                              2670
4  CUST005877       PLAN03                               209
5  CUST003762       PLAN02                               261
6  CUST007892       PLAN02                               303
7  CUST002317       PLAN01                              2261
8  CUST005609       PLAN02                              1274
9  CUST000519       PLAN03                               434


In [64]:
customers["has_failed_payment"] = (
    customers["failed_payment_count"] > 0
).astype(int)

print(customers[[
    "customer_id",
    "failed_payment_count",
    "payment_failure_rate",
    "has_failed_payment"
]].head(10))

  customer_id  failed_payment_count  payment_failure_rate  has_failed_payment
0  CUST000744                   2.0              0.133333                   1
1  CUST001502                   0.0              0.000000                   0
2  CUST005383                   1.0              0.066667                   1
3  CUST004708                   1.0              0.066667                   1
4  CUST005877                   1.0              0.166667                   1
5  CUST003762                   1.0              0.125000                   1
6  CUST007892                   0.0              0.000000                   0
7  CUST002317                   0.0              0.000000                   0
8  CUST005609                   0.0              0.000000                   0
9  CUST000519                   0.0              0.000000                   0


In [65]:
print("Customers shape:", customers.shape)

print("\nMissing values:")
print(customers.isna().sum())

print("\nDuplicate customer IDs:",
      customers["customer_id"].duplicated().sum())

Customers shape: (8000, 35)

Missing values:
customer_id                            0
first_name                             0
last_name                              0
age                                    0
gender                                 0
country                                0
city                                   0
registration_date                      0
acquisition_channel                    0
customer_segment                       0
preferred_language                     0
tenure_days                            0
total_watch_time_minutes               0
total_viewing_sessions                 0
average_completion_percentage          0
average_rating                         0
feedback_count                         0
support_ticket_count                   0
average_support_satisfaction        4460
has_support_satisfaction               0
successful_payment_count               0
total_successful_payment_amount        0
failed_payment_count                   0
subscription

In [66]:
print(
    customers[[
        "customer_id",
        "average_support_satisfaction",
        "has_support_satisfaction"
    ]].head(10)
)

print("\nCustomers with satisfaction score:",
      customers["has_support_satisfaction"].sum())

print("Customers without satisfaction score:",
      (customers["has_support_satisfaction"] == 0).sum())

  customer_id  average_support_satisfaction  has_support_satisfaction
0  CUST000744                           NaN                         0
1  CUST001502                           NaN                         0
2  CUST005383                           NaN                         0
3  CUST004708                           NaN                         0
4  CUST005877                           NaN                         0
5  CUST003762                           NaN                         0
6  CUST007892                           NaN                         0
7  CUST002317                           NaN                         0
8  CUST005609                           2.0                         1
9  CUST000519                           4.0                         1

Customers with satisfaction score: 3540
Customers without satisfaction score: 4460


In [67]:
print("Negative values check:")

numeric_features = [
    "tenure_days",
    "total_watch_time_minutes",
    "total_viewing_sessions",
    "average_completion_percentage",
    "feedback_count",
    "support_ticket_count",
    "successful_payment_count",
    "total_successful_payment_amount",
    "failed_payment_count",
    "subscription_count",
    "current_subscription_tenure_days",
    "average_watch_time_per_session",
    "unique_titles_watched",
    "active_viewing_days",
    "payment_failure_rate",
    "unique_genres_watched"
]

for column in numeric_features:
    print(
        column,
        "→",
        (customers[column] < 0).sum()
    )

Negative values check:
tenure_days → 0
total_watch_time_minutes → 0
total_viewing_sessions → 0
average_completion_percentage → 0
feedback_count → 0
support_ticket_count → 0
successful_payment_count → 0
total_successful_payment_amount → 0
failed_payment_count → 0
subscription_count → 0
current_subscription_tenure_days → 0
average_watch_time_per_session → 0
unique_titles_watched → 0
active_viewing_days → 0
payment_failure_rate → 0
unique_genres_watched → 0


In [68]:
FEATURE_FILE = CLEANED_DIR / "customer_features.csv"

customers.to_csv(
    FEATURE_FILE,
    index=False
)

print("Feature dataset saved successfully.")
print("Shape:", customers.shape)
print("File:", FEATURE_FILE)

Feature dataset saved successfully.
Shape: (8000, 35)
File: ..\data\cleaned\customer_features.csv
